In [5]:
import os
import shutil
import kagglehub
import pandas as pd

# 1. Download the complete dataset from Kaggle Hub
download_path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print(f"Files downloaded by kagglehub in: {download_path}")

# 2. Copy the .csv files to our local data/raw/ folder
target_dir = '../data/raw'
os.makedirs(target_dir, exist_ok=True)

for file_name in os.listdir(download_path):
    if file_name.endswith('.csv'):
        src_file = os.path.join(download_path, file_name)
        dst_file = os.path.join(target_dir, file_name)
        shutil.copy(src_file, dst_file)

print("All CSV files have been successfully copied to data/raw/!\n")

# 3. Inspect the copied datasets
csv_files = [f for f in os.listdir(target_dir) if f.endswith('.csv')]
print(f"Total CSV files in data/raw: {len(csv_files)}\n")

for file in sorted(csv_files):
    df = pd.read_csv(os.path.join(target_dir, file))
    print(f"📄 File: {file}")
    print(f"   Shape: {df.shape} (Rows, Columns)")
    print("-" * 50)

Files downloaded by kagglehub in: C:\Users\jayzi\.cache\kagglehub\datasets\olistbr\brazilian-ecommerce\versions\2
All CSV files have been successfully copied to data/raw/!

Total CSV files in data/raw: 9

📄 File: olist_customers_dataset.csv
   Shape: (99441, 5) (Rows, Columns)
--------------------------------------------------
📄 File: olist_geolocation_dataset.csv
   Shape: (1000163, 5) (Rows, Columns)
--------------------------------------------------
📄 File: olist_order_items_dataset.csv
   Shape: (112650, 7) (Rows, Columns)
--------------------------------------------------
📄 File: olist_order_payments_dataset.csv
   Shape: (103886, 5) (Rows, Columns)
--------------------------------------------------
📄 File: olist_order_reviews_dataset.csv
   Shape: (99224, 7) (Rows, Columns)
--------------------------------------------------
📄 File: olist_orders_dataset.csv
   Shape: (99441, 8) (Rows, Columns)
--------------------------------------------------
📄 File: olist_products_dataset.csv
  

In [6]:
import os
import pandas as pd

# Define the relative path to raw data
RAW_DATA_PATH = "../data/raw"

# List and sort all CSV files in the directory
csv_files = sorted([f for f in os.listdir(RAW_DATA_PATH) if f.endswith(".csv")])

print(f"📊 OLIST DATASETS TECHNICAL INSPECTION ({len(csv_files)} tables)\n" + "=" * 60)

# Iterate through each CSV file to inspect shape, duplicates, and missing values
for file in csv_files:
    file_path = os.path.join(RAW_DATA_PATH, file)
    df = pd.read_csv(file_path)
    
    # Filter columns that contain missing values
    missing_values = df.isnull().sum()[df.isnull().sum() > 0].to_dict()
    
    print(f"\n📄 File: {file}")
    print(f"   • Rows: {df.shape[0]:,}")
    print(f"   • Columns: {df.shape[1]}")
    print(f"   • Exact Duplicates: {df.duplicated().sum()}")
    print(f"   • Missing Values: {missing_values}")
    print("-" * 60)

📊 OLIST DATASETS TECHNICAL INSPECTION (9 tables)



📄 File: olist_customers_dataset.csv
   • Rows: 99,441
   • Columns: 5
   • Exact Duplicates: 0
   • Missing Values: {}
------------------------------------------------------------

📄 File: olist_geolocation_dataset.csv
   • Rows: 1,000,163
   • Columns: 5
   • Exact Duplicates: 261831
   • Missing Values: {}
------------------------------------------------------------

📄 File: olist_order_items_dataset.csv
   • Rows: 112,650
   • Columns: 7
   • Exact Duplicates: 0
   • Missing Values: {}
------------------------------------------------------------

📄 File: olist_order_payments_dataset.csv
   • Rows: 103,886
   • Columns: 5
   • Exact Duplicates: 0
   • Missing Values: {}
------------------------------------------------------------

📄 File: olist_order_reviews_dataset.csv
   • Rows: 99,224
   • Columns: 7
   • Exact Duplicates: 0
   • Missing Values: {'review_comment_title': 87656, 'review_comment_message': 58247}
------------------------------------------------------------

📄 File: o

In [10]:
import os
import pandas as pd

RAW_DATA_PATH = "../data/raw"

# Load all datasets into a dictionary
data = {}
csv_files = [f for f in os.listdir(RAW_DATA_PATH) if f.endswith(".csv")]

for file in csv_files:
    key = file.replace(".csv", "").replace("olist_", "").replace("_dataset", "")
    data[key] = pd.read_csv(os.path.join(RAW_DATA_PATH, file))

# 1. Clean Geolocation: Remove exact duplicates and drop NaN zip codes if any
data["geolocation"] = data["geolocation"].drop_duplicates()

# 2. Convert timestamp columns in orders to datetime format
datetime_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in datetime_cols:
    data["orders"][col] = pd.to_datetime(data["orders"][col], errors="coerce")

# 3. Fill missing product categories with 'unknown'
data["products"]["product_category_name"] = data["products"]["product_category_name"].fillna("unknown")

# Summary of clean datasets
print("✅ DATA CLEANING & PARSING COMPLETE\n" + "=" * 50)
print(f"• Geolocation deduplicated rows: {data['geolocation'].shape[0]:,}")
print(f"• Orders datetime conversion check:\n{data['orders'][datetime_cols].dtypes}\n")
print(f"• Missing categories in products: {data['products']['product_category_name'].isnull().sum()}")

✅ DATA CLEANING & PARSING COMPLETE
• Geolocation deduplicated rows: 738,332
• Orders datetime conversion check:
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

• Missing categories in products: 0


In [ ]:
# 1. Merge orders with customers
df_merged = data["orders"].merge(data["customers"], on="customer_id", how="inner")

# 2. Merge with order_items
df_merged = df_merged.merge(data["order_items"], on="order_id", how="left")

# 3. Merge with products
df_merged = df_merged.merge(data["products"], on="product_id", how="left")

# 4. Merge with category name translations
df_merged = df_merged.merge(data["product_category_name_translation"], on="product_category_name", how="left")

# 5. Merge with order_payments
df_merged = df_merged.merge(data["payments"], on="order_id", how="left")

# 6. Merge with order_reviews
df_merged = df_merged.merge(data["reviews"], on="order_id", how="left")

print("✅ DATASET MERGE COMPLETED SUCCESSFULLY\n" + "=" * 50)
print(f"• Final Merged Shape: {df_merged.shape[0]:,} rows | {df_merged.shape[1]} columns")
print(f"• Sample Columns:\n{list(df_merged.columns[:10])}\n")
print(f"• Total Revenue (Price): ${df_merged['price'].sum():,.2f}")
print(f"• Total Freight Value: ${df_merged['freight_value'].sum():,.2f}")